<a href="https://colab.research.google.com/github/Alenushka2013/ML_for_people_tasks/blob/main/HW_6_Using_prompts_and_agents_in_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### Завдання 1: Виклик LLM з базовим промптом

Створіть можливість викликати LLM зі звичайним текстовим промптом.

Промпт має дозвляти отримати інформацію простою мовою на певну тему. В цьому завданні ми хочемо дізнатись про тему "Квантові обчислення".

Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

Обмежте відповідь до 200 символів і пропишіть в промпті аби відповідь була короткою (це зекономить Вам час і гроші на згенеровані токени).

В якості LLM можна скористатись як моделлю з HugginFace (рекомендую Mistral), так і ChatGPT4 або ChatGPT3. В обох випадках треба імпортувати потрібну "обгортку" (тобто клас, який дозволить ініціювати модель) з LangChain для виклику LLM за API, а також зчитати особистий токен з файла, наприклад, `creds.json`, який розміщений у Вас локально і Ви НЕ здаєте його в ДЗ і НЕ комітите в git 😏

Встановіть своє значення температури на свій розсуд (тут немає правильного чи неправильного значення) і напишіть, чому ви обрали саме таке значення для цього завдання.  

Запити можна робити як українською, так і англійською - орієнтуйтесь на те, де і чи хочете ви потім лишити цей проєкт і відповідна яка мова буде пасувати більше. В розвʼязках промпти - українською.

In [ ]:
!pip install openai

In [ ]:
!pip install langchain_openai

In [ ]:
import json

with open("creds.json") as file:
    creds = json.load(file)

In [ ]:
import openai

openai.api_key = creds["OPENAI_API_KEY"]

prompt = """
  Provide a brief definition of quantum computing.
  List the key benefits of this technology and give an example of current research in this area.
  Limit your answer to 200 characters."""

response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are an expert on scientific topics."},
        {"role": "user", "content": prompt}
    ],
    max_tokens=200, # Обмеження довжини
    temperature=0.1   # Можна змінювати для більшої креативності
)

print(response.choices[0].message.content)

Quantum computing uses quantum bits (qubits) to perform calculations at unprecedented speeds. Benefits include faster problem-solving, enhanced security, and optimization. Current research includes quantum algorithms for drug discovery.


Температура рівна 0.1 відповідає мінімальній креативності мовної моделі. Визначення наукового терміну хотілося б отримати точним, без зайвих вигадок.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модель) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import os

# Встановлення ключа OpenAI
os.environ["OPENAI_API_KEY"] = creds["OPENAI_API_KEY"]

# Ініціалізуємо LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1, max_tokens=200)

# Створюємо шаблон промпта з параметром {topic}
template = """Provide a brief definition of the topic "{topic}".
   State the key benefits of this technology and provide an example of current research in this area.
   The answer should be concise and informative, without the use of any formatting symbols (e.g. asterisks, hash marks, etc.)."""

prompt_template = PromptTemplate(input_variables=["topic"], template=template)

# Створюємо ланцюг (chain), який поєднує шаблон і LLM
chain = LLMChain(llm=llm, prompt=prompt_template)

# Список тем для обробки
topics = ["Bayesian methods in machine learning",
          "Transformers in machine learning",
          "Explainable AI"]

# Запускаємо ланцюг для кожної теми
for topic in topics:
    print(f"--- Topic: {topic} ---")
    response = chain.invoke(topic)
    print(response['text'])
    print("\n")

--- Topic: Bayesian methods in machine learning ---
Bayesian methods in machine learning refer to a set of statistical techniques that apply Bayes' theorem to update the probability of a hypothesis as more evidence or information becomes available. These methods allow for the incorporation of prior knowledge and uncertainty into the modeling process, enabling more robust decision-making.

Key benefits of Bayesian methods include the ability to quantify uncertainty in predictions, the flexibility to incorporate prior information, and the capability to update models dynamically as new data is observed. This makes them particularly useful in scenarios where data is scarce or noisy.

An example of current research in this area is the application of Bayesian deep learning, which combines deep learning techniques with Bayesian inference to improve model robustness and uncertainty estimation in tasks such as image classification and natural language processing. Researchers are exploring ways 



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [ ]:
!pip install -q langchain_community duckduckgo_search

In [ ]:
!pip install requests==2.32.4

In [ ]:
!pip install -U ddgs

In [ ]:
!pip install arxiv

In [ ]:
!pip install -q google-search-results langchain-community langchain_experimental

In [ ]:
!pip -q install langchain langchain_openai huggingface_hub openai langchain_huggingface

In [ ]:
! pip install langchain_mistralai

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

In [ ]:
import json
import os

with open('creds.json') as file:
  creds = json.load(file)

os.environ["OPENAI_API_KEY"] = creds["OPEN_AI_16_09"]
os.environ["SERPAPI_API_KEY"] = creds["SERPARI_16_09"]
os.environ["MISTRAL_API_KEY"] = creds["Mistral_16_09"]
os.environ["HF_TOKEN"] = creds["HUGGINGFACEHUB_API_TOKEN"]

In [ ]:
from langchain import hub
from langchain.agents import load_tools
from langchain.agents import Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain.chains import LLMMathChain
from langchain_experimental.utilities import PythonREPL
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI

overal_temperature = 0.1
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=overal_temperature)
llm = ChatMistralAI(model="open-mistral-7b",temperature=overal_temperature, mistral_api_key=creds['Mistral_16_09'])
search_tool = Tool(name="Researcher",
                func=search.run,
                description="A useful tool for searching last scientific publications on the Internet.")

In [ ]:
template = """
You are a specialized agent for searching scientific publications on the Internet or in databases of scientific publications.
Your goal is to find and analyze scientific articles on given topics since May 2025.

Each of the puvlication contains:
 - the title
 - authors
 - a brief description (2-3 sentences).

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""

In [ ]:
tools = []
tools.append(search_tool)
custom_prompt = PromptTemplate.from_template(template)
agent = create_react_agent(llm, tools, custom_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=15)

In [ ]:
agent_executor.invoke({'input': "Find 5 recent scientific publications on the topic of artificial intelligence."})

Токени закінчилися ще до того, як вдалося отримати задовільний результат. Далі представлені альтернативні способи вирішення завдання.

In [ ]:
import os
import json
import time
import httpx
from tenacity import retry, stop_after_attempt, wait_fixed

from langchain import hub
from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI

# завантаження ключів
with open('creds.json') as file:
    creds = json.load(file)

os.environ["OPENAI_API_KEY"] = creds["OPEN_AI_16_09"]
os.environ["SERPAPI_API_KEY"] = creds["SERPARI_16_09"]
os.environ["MISTRAL_API_KEY"] = creds["Mistral_16_09"]
os.environ["HF_TOKEN"] = creds["HUGGINGFACEHUB_API_TOKEN"]

#  Модель
llm = ChatMistralAI(
    model="open-mistral-7b",
    temperature=0.1,
    mistral_api_key=creds['Mistral_16_09']
)

#  Тул для пошуку
from langchain.utilities import SerpAPIWrapper
search = SerpAPIWrapper()
search_tool = Tool(
    name="Researcher",
    func=search.run,
    description="Searches for recent scientific publications on the Internet"
)

#  промпт
template = """
You are a specialized agent for searching scientific publications on the Internet.
Your goal is to find and analyze scientific articles on given topics since May 2025.

Each publication must include:
 - the title
 - authors
 - a brief description (2-3 sentences).

Use the tools below to answer the question.

{tools}

Follow this format:

Question: the input question
Thought: reasoning about what to do
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: result of the action
... repeat if needed
Thought: I now know the final answer
Final Answer: the list of publications

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""
custom_prompt = PromptTemplate.from_template(template)

#  Агент
agent = create_react_agent(llm, [search_tool], custom_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=False,
    handle_parsing_errors=True,  #  прибирає OutputParserException
    max_iterations=10
)

#  Функція з обмеженням швидкості
@retry(stop=stop_after_attempt(3), wait=wait_fixed(2))
def safe_invoke(query: str):
    try:
        # пауза для тарифу (1 req/sec)
        time.sleep(1)
        result = agent_executor.invoke({"input": query})
        return result["output"] if "output" in result else result
    except httpx.HTTPStatusError as e:
        if e.response.status_code == 429:
            raise e  # tenacity повторить
        else:
            raise

# Виклик та отримання результатів
res = safe_invoke("Find 5 recent scientific publications on the topic of artificial intelligence.")
print("\n=== RESULT ===")
print(res)



=== RESULT ===
Here are 5 recent scientific publications on the topic of artificial intelligence:

1. Title: Deep Learning for Image Recognition: Advances and Challenges
   Authors: John Doe, Jane Smith, and Robert Johnson
   Description: This paper discusses the latest advancements and challenges in deep learning for image recognition, focusing on convolutional neural networks and their applications in various fields.

2. Title: Exploring the Ethical Implications of Artificial Intelligence
   Authors: Alice Brown and Charles Green
   Description: This study investigates the ethical implications of artificial intelligence, discussing issues such as bias, privacy, and accountability in AI systems.

3. Title: Reinforcement Learning in Robotics: A Comprehensive Review
   Authors: David Lee and Michael Chen
   Description: This review paper provides an overview of reinforcement learning in robotics, discussing its applications in robot navigation, manipulation, and learning from demonstra

In [ ]:
import feedparser
from urllib.parse import quote

def arxiv_search(topic, max_results=5):
    """
    Шукає останні публікації на arXiv за темою topic.
    Повертає список словників з назвою, авторами та описом.
    """
    base_url = "http://export.arxiv.org/api/query?"
    encoded_topic = quote(topic)  # <-- кодуємо тему у URL
    search_query = f"search_query=all:{encoded_topic}&start=0&max_results={max_results}&sortBy=submittedDate&sortOrder=descending"
    url = base_url + search_query
    feed = feedparser.parse(url)

    results = []
    for entry in feed.entries:
        title = entry.title.replace("\n", " ").strip()
        authors = ", ".join([author.name for author in entry.authors])
        summary = entry.summary.replace("\n", " ").strip()
        results.append({
            "Topic": topic,
            "Title": title,
            "Authors": authors,
            "Summary": summary
        })
    return results

topics = [
    "Artificial Intelligence",
    "machine learning algorithms",
    "Explainable AI"
]

all_results = []
for t in topics:
    all_results.extend(arxiv_search(t))

# Вивід результатів
for topic in topics:
    print(f"=== Тема: {topic} ===")
    for r in [x for x in all_results if x["Topic"] == topic]:
        print(f"Назва: {r['Title']}")
        print(f"Автори: {r['Authors']}")
        print(f"Опис: {r['Summary']}\n")
    print("-" * 80)

=== Тема: Artificial Intelligence ===
Назва: Apertus: Democratizing Open and Compliant LLMs for Global Language   Environments
Автори: Alejandro Hernández-Cano, Alexander Hägele, Allen Hao Huang, Angelika Romanou, Antoni-Joan Solergibert, Barna Pasztor, Bettina Messmer, Dhia Garbaya, Eduard Frank Ďurech, Ido Hakimi, Juan García Giraldo, Mete Ismayilzada, Negar Foroutan, Skander Moalla, Tiancheng Chen, Vinko Sabolčec, Yixuan Xu, Michael Aerni, Badr AlKhamissi, Ines Altemir Marinas, Mohammad Hossein Amani, Matin Ansaripour, Ilia Badanin, Harold Benoit, Emanuela Boros, Nicholas Browning, Fabian Bösch, Maximilian Böther, Niklas Canova, Camille Challier, Clement Charmillot, Jonathan Coles, Jan Deriu, Arnout Devos, Lukas Drescher, Daniil Dzenhaliou, Maud Ehrmann, Dongyang Fan, Simin Fan, Silin Gao, Miguel Gila, María Grandury, Diba Hashemi, Alexander Hoyle, Jiaming Jiang, Mark Klein, Andrei Kucharavy, Anastasiia Kucherenko, Frederike Lübeck, Roman Machacek, Theofilos Manitaras, Andreas Marfu



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [ ]:
import time
from tenacity import retry, stop_after_attempt, wait_fixed
from langchain_openai import ChatOpenAI

# Дані по експорту
exports = {
    2021: 200,
    2022: 190,
    2023: 210,
    2024: 220
}

# Розрахунок середнього зростання
growth_rates = [(exports[year] - exports[year-1]) / exports[year-1]
                for year in range(2022, 2025)]
average_growth_rate = sum(growth_rates) / len(growth_rates)

# Прогноз на 2025
predicted_2025_export = round(exports[2024] * (1 + average_growth_rate), 2)

# Налаштування LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# Retry, щоб не впасти по RateLimit
@retry(stop=stop_after_attempt(5), wait=wait_fixed(25))
def generate_business_report(prediction, growth_rate):
    query = f"""
    Ми експортуємо апельсини з Бразилії.
    Дані: {exports}.
    Середнє зростання: {growth_rate:.2%}.
    Прогноз на 2025: {prediction} т.

    Завдання:
    - Поясни прогноз на 2025 рік (експорт апельсинів).
    - Врахуй можливий вплив погоди в Бразилії та інфляції у світі.
    - Дай коротку бізнес-оцінку у 3-4 реченнях.
    """
    return llm.invoke(query)

#  Виклик
response = generate_business_report(predicted_2025_export, average_growth_rate)
print("=== Прогноз бізнес-аналітики ===")
print(response.content)


=== Прогноз бізнес-аналітики ===
Прогноз на 2025 рік, який становить 227.54 тонни експорту апельсинів з Бразилії, базується на середньому зростанні експорту за останні роки, що складає 3.43%. Це зростання може бути підкріплене стабільним попитом на апельсини на міжнародному ринку. Однак важливо врахувати, що погодні умови в Бразилії можуть суттєво вплинути на врожайність: посухи або надмірні опади можуть зменшити обсяги виробництва. Крім того, глобальна інфляція може вплинути на витрати на виробництво та транспортування, що, в свою чергу, може позначитися на ціні та конкурентоспроможності бразильських апельсинів.

У бізнес-оцінці можна зазначити, що, незважаючи на позитивний прогноз, компанії слід бути готовими до можливих ризиків, пов'язаних із погодними умовами та економічними змінами. Рекомендується розробити стратегії для мінімізації ризиків, такі як диверсифікація постачань та укладання контрактів на фіксовані ціни. Це дозволить зберегти стабільність бізнесу в умовах непередбачува

**Висновки:**
Цікаві інструменти, які варто вивчити глибше, але через обмеження доступу не завжди виходить досягти очікуваного результату.

In [32]:
import os
import json

# завантаження ключів
with open('creds.json') as file:
    creds = json.load(file)

os.environ["OPENAI_API_KEY"] = creds["OPEN_AI_16_09"]
os.environ["SERPAPI_API_KEY"] = creds["SERPARI_16_09"]
os.environ["MISTRALAI_API_KEY"] = creds["Mistral_16_09"]

In [34]:
from langchain.agents import initialize_agent, Tool, AgentType
from langchain_experimental.utilities import PythonREPL
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_mistralai import ChatMistralAI
import os
import json

llm = ChatMistralAI(model="mistral-small-latest", temperature=0)

# Інструменти
python_tool = PythonREPL()
search_tool = DuckDuckGoSearchRun()

tools = [
    Tool(
        name="Python REPL",
        func=python_tool.run,
        description="Використовується для аналізу даних та прогнозу продажів."
    ),
    Tool(
        name="Web Search",
        func=search_tool.run,
        description="Шукати актуальні дані про погоду, економіку та попит на апельсини."
    )
]

# Створення агента
agent_executor = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    # Ось ключова зміна
    handle_parsing_errors=True
)

# Запуск
user_input = """
Ми експортуємо апельсини з Бразилії.
В 2021 експортували 200т, в 2022 - 190т, в 2023 - 210т, в 2024 який ще не закінчився - 220т.
Зроби оцінку скільки ми зможемо експортувати апельсинів в 2025 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
Якщо прогноз неможливий, напиши, чого бракує.
"""

result = agent_executor.run(user_input)
print(result)



> Entering new AgentExecutor chain...


Parsing LLM output produced both a final answer and a parse-able action:: To answer this question, I need to gather information about the current weather conditions in Brazil, the global economic situation, and the demand for oranges worldwide. Additionally, I should analyze the historical export data to identify any trends or patterns.

Action: Web Search
Action Input: "Current weather conditions in Brazil affecting orange production"
Action Input: "Global economic situation and demand for oranges"
Action Input: "Historical trends in orange exports from Brazil"

Observation: (Results from the web search will be gathered here)

Thought: After gathering the necessary data, I will analyze the historical export data to identify any trends or patterns. Then, I will incorporate the current weather conditions and global economic situation to make an informed prediction for the 2025 orange export volume.

Action: Python REPL
Action Input: Analyze the historical export data and make a predicti

Parsing LLM output produced both a final answer and a parse-able action:: It seems the previous attempt to gather data and analyze it was incomplete. Let me re-evaluate the approach and try again.

**Thought:**
To provide an accurate forecast for orange exports in 2025, I need:
1. **Historical data analysis** (already provided: 2021-2024 exports).
2. **Current weather conditions in Brazil** (to assess potential crop impacts).
3. **Global economic trends and demand for oranges** (to estimate market factors).

Since the previous web search did not yield results, I will refine the search queries and ensure the Python REPL analysis is properly structured.

**Action:** Web Search
**Action Input:**
1. "Current weather conditions in Brazil affecting orange production 2024"
2. "Global demand for oranges 2024 economic outlook"
3. "Orange export trends Brazil 2021-2024"

**Observation:**
*(Awaiting results from the refined web search.)*

**Thought:**
If the web search fails again, I will proceed

**Висновки:**

 - агент правильно визначив, що йому потрібен інструмент Python REPL для аналізу числових даних, а також інструмент веб-пошуку для інформації про погоду та світову економіку;
 - агент використав модель лінійної регресії для прогнозування обсягу експорту на 2025 рік на основі історичних даних;
 - в остаточній відповіді агента зазначено, що він не зміг отримати зовнішні дані(про погодні умови та економічні тенденції) - над цим ще потрібно працювати.